# CRE3 - Assignment 2: Parametric study of $CO_2$/$CO$ hydrogenation and DME synthesis

Group 6: 

Julian Stierstorfer (1160552)\
Ivana Garzon Casanova ()\
Venkata Uda ()

## 1. Introduction

Assumptions: General approach for the reaction network, continuous operation.

## 2. Reaction network and key reactions

The reaction system that describes the hydrogenation of $CO_2$ and $CO$ as well as the synthesis of DME is given as follows:

\begin{align}
CO_2 + 3H_2 &\overset{R_1}{\rightleftharpoons} CH_3OH + H_2O\\
CO + 2H_2 &\overset{R_2}{\rightleftharpoons} CH_3OH\\
CO_2 + H_2 &\overset{R_3}{\rightleftharpoons} CO + H_2O\\
2CH_3OH &\overset{R_4}{\rightleftharpoons} CH_3OCH_3 + H_2O
\end{align}

Equations (1) and (2) describe the hydrogenation of $CO_2$ and $CO$ while forming methanol ($CH_3OH$). Reaction (3) shows the reverse water-gas shift reaction, while reaction (4) describes the formation of DME from methanol.

### 2.1 Matrix of stoichiometric coefficients

From the reaction equations the Matrix of stoichiometric components can be derived by sorting all components and reactions:

\begin{equation}
  \underline{N} =
    \begin{bmatrix}
          & R_1 & R_2 & R_3 & R_4\\
      CO_2      & -1 & 0  & -1 & 0\\
      H_2       & -3 & -2 & -1 & 0\\
      CH_3OH    & 1  & 1  & 0  & -2\\
      H_2O      & 1  & 0  & 1  & 1\\
      CO        & 0  & -1 & 1  & 0\\
      CH_3OCH_3 & 0  & 0  & 0  & 1
    \end{bmatrix}
\end{equation}

### 2.2 Determining key reactions and components
The number of key reactions and components $R_{\nu}$ is calculated in order to obtain a minimum independent description of the reaction network. Once key reactions and components are known, the linear independent part of the reaction network can be used for a simpler calculation. In practice this also means that by measuring the concentration of key components at a given time, concentration of all other components can be calculated. 

The number of key reactions and components can be calculated as follows [1]:
\begin{equation}
R_\nu = \operatorname{rank}(\underline{N})
\end{equation} 

The rank of $\underline{N}$ can in principle be calculated by applying the Gauss algorithm by hand however this can be done numerically using the `matrix_rank` function by the `linalg`-package by `numpy`. Therefore the initial matrix $\underline{N}_{init}$ is implemented and its rank is calculated.

In [6]:
# importing needed packages
import numpy as np # general numpy package
from numpy.linalg import matrix_rank, det, inv # linear algebra package for matrix rank, determinant and inverse
import matplotlib.pyplot as plt # plotting package


# defining labels for rows (components) and  columns (reactions)
components = ["CO2", "H2", "CH3OH", "H2O", "CO", "CH3OCH3"]
reactions = ["R1", "R2", "R3", "R4"]

N_init = np.array([
    [-1,  0, -1,  0],   # CO2
    [-3, -2, -1,  0],   # H2
    [ 1,  1,  0, -2],   # CH3OH
    [ 1,  0,  1,  1],   # H2O
    [ 0, -1,  1,  0],   # CO
    [ 0,  0,  0,  1]    # CH3OCH3
], dtype=float) # initial matrix of stoichiometric coefficients for the reactions

# Calculate the rank of the stoichiometric matrix
rank_N = matrix_rank(N_init)
print(f"Rank of the stoichiometric matrix N: {rank_N}")



Rank of the stoichiometric matrix N: 3


As obtained from the numerical calculation the rank of the matrix is found. $R_{\nu} = 3$. Therefore, three key components are required. Their selection should consider both practical measurability and chemical relevance, meaning that the chosen components should be experimentally accessible and meaningful for describing the reaction network.

In this case the key components are choosen to be $CO_2$, $CH_3OH$ and $DME$. On the one hand $CO_2$ is one of the carbon feeding species that takes place in hydrogenation, on the other hand $CH_3OH$ and $DME$ represent a critical intermediate as well as the product component.

As well as taking into account key components, key reactions can be determined as well. According to the rank of $\underline{N}$, 3 linear independent key reactions are sufficient to describe the reaction system. After further investigation it can be found that $R_3 = R_1 - R_2$. Therefore the key reactions of the system are $R_1, R_2$ and $R_4$.

Thus the matrix of stoichiometric components can be rearranged according to the key components and reactions:

In [7]:
# Rearranged matrix
# key components first: CO2, CH3OH, CH3OCH3
# remaining components: H2, H2O, CO
row_order = [0, 2, 5, 1, 3, 4]

# independent key reactions first: R1, R2, R4
# dependent reaction R3 last
col_order = [0, 1, 3, 2]

N = N_init[row_order, :][:, col_order]

component_order = ["CO2", "CH3OH", "CH3OCH3", "H2", "H2O", "CO"]
reaction_order = ["R1", "R2", "R4", "R3"]

print("Rearranged matrix N:")
print(N)

print("\nComponent order:")
print(component_order)

print("\nReaction order:")
print(reaction_order)

print("\nRank of rearranged matrix:")
print(matrix_rank(N))

Rearranged matrix N:
[[-1.  0.  0. -1.]
 [ 1.  1. -2.  0.]
 [ 0.  0.  1.  0.]
 [-3. -2.  0. -1.]
 [ 1.  0.  1.  1.]
 [ 0. -1.  0.  1.]]

Component order:
['CO2', 'CH3OH', 'CH3OCH3', 'H2', 'H2O', 'CO']

Reaction order:
['R1', 'R2', 'R4', 'R3']

Rank of rearranged matrix:
3


The rearranged matrix is now given as:

\begin{equation}
  \underline{N} =
    \begin{bmatrix}
          & R_1 & R_2 & R_4 & R_3\\
      CO_2      & -1 & 0  & 0  & -1\\
      CH_3OH    & 1  & 1  & -2 & 0\\
      CH_3OCH_3 & 0  & 0  & 1  & 0\\
      H_2       & -3 & -2 & 0  & -1\\
      H_2O      & 1  & 0  & 1  & 1\\
      CO        & 0  & -1 & 0  & 1
    \end{bmatrix}
\end{equation}

This matrix can now be partitioned into four submatrices:

\begin{equation}
\underline{N} =
\begin{bmatrix}
\underline{N}_{1,1} & \underline{N}_{1,2}\\
\underline{N}_{2,1} & \underline{N}_{2,2}
\end{bmatrix} =
\left[
\begin{array}{ccc|c}
-1 & 0  & 0  & -1\\
 1 & 1  & -2 & 0\\
 0 & 0  & 1  & 0\\
\hline
-3 & -2 & 0  & -1\\
 1 & 0  & 1  & 1\\
 0 & -1 & 0  & 1
\end{array}
\right]
\end{equation}





In [8]:
# Calculate the rank of the rearranged matrix
R_nu = matrix_rank(N)

# Split matrix into submatrices
N_11 = N[:R_nu, :R_nu]
N_12 = N[:R_nu, R_nu:]
N_21 = N[R_nu:, :R_nu]
N_22 = N[R_nu:, R_nu:]

print("\nN_11:")
print(N_11)

print("\nN_12:")
print(N_12)

print("\nN_21:")
print(N_21)

print("\nN_22:")
print(N_22)




N_11:
[[-1.  0.  0.]
 [ 1.  1. -2.]
 [ 0.  0.  1.]]

N_12:
[[-1.]
 [ 0.]
 [ 0.]]

N_21:
[[-3. -2.  0.]
 [ 1.  0.  1.]
 [ 0. -1.  0.]]

N_22:
[[-1.]
 [ 1.]
 [ 1.]]


### 2.3 Reaction Extent
The stoichiometric changes during the reaction can be investigated by using the reaction extent $\xi_j$ which can be derived from the generall mass balance. The reaction extent takes into account the difference in molar flow between input and output stream for every component:

\begin{align}
  \underline{\dot n}_{out}-\underline{\dot n}_{in} =\Delta\underline{\dot n} = \underline{N}\,\underline{\dot\xi}
\end{align}

$\Delta\underline{\dot n}$ contains the overall conversion of the key components ($\Delta\underline{\dot n_1}$) as well as the non-key components ($\Delta\underline{\dot n_2}$)

\begin{equation}
  \Delta\underline{\dot n}=
    \begin{bmatrix}
      \Delta\underline{\dot n}_{1}\\
      \\
      \Delta\underline{\dot n}_{2}
    \end{bmatrix}%^1_2
\end{equation}

While the conversion of the key components is known (e.g. from measurements) the conversion of the non-key components can be calculated:

\begin{equation}
  \Delta\underline{\dot n}_{2} = \underline{N}_{21}\,\underline{N}_{11}^{-1}\,\Delta\underline{\dot n}_{1}
\end{equation}

Therefore, $\underline{N}_{11}$ has to be invertible in the first place. This can be checked by calculating the determinant of the matrix which is done by using the `det`-function of `numpy`'s `linalg`-package:

In [ ]:
print("\ndet(N_11):")
print(det(N_11))


det(N_11):
-1.0


Since $\det\left(\underline{N}_{1,1}\right) \neq 0$, this is guaranteed.

### 2.4 Conversion

For the given system an input molar flow for all components is assumed:

\begin{equation}
\dot{\underline{n}}_{in} =
\begin{bmatrix}
10.0 & CO_2\\
0.2  & CH_3OH\\
0.3  & CH_3OCH_3\\
30.0 & H_2\\
0.69  & H_2O\\
2.0  & CO
\end{bmatrix}
\ \mathrm{mol\,s^{-1}}
\end{equation}

The conversion of the key components wcan be measured and is in this case assumed to be:

\begin{equation}
\Delta \dot{\underline{n}}_1 =
\begin{bmatrix}
-5.5 & CO_2\\
 1.0 & CH_3OH\\
 1.0 & CH_3OCH_3
\end{bmatrix}
\ \mathrm{mol\,s^{-1}}
\end{equation}

The conversion of the non-key components $\Delta\underline{\dot n_2}$ as well as the reaction extent $\dot{\underline{\xi}}$ of the key reactions can now be calculated:

In [13]:
# New inlet molar flow vector
# order: CO2, CH3OH, CH3OCH3, H2, H2O, CO
n_in = np.array([
    10.0,   # CO2
     0.2,   # CH3OH
     0.3,   # CH3OCH3
    30.0,   # H2
     0.69,  # H2O
     2.0    # CO
])

# Measured / assumed change of key-component molar flows
# order: CO2, CH3OH, CH3OCH3
delta_n_1 = np.array([
    -5.5,   # CO2
     1.0,   # CH3OH
     1.0    # CH3OCH3
])

# Calculate change of non-key-component molar flows
# Δn_2 = N_21 @ inv(N_11) @ Δn_1
delta_n_2 = N_21 @ np.linalg.inv(N_11) @ delta_n_1

# Calculate reaction extent vector
# ξ = inv(N_11) @ Δn_1
reaction_extent = inv(N_11) @ delta_n_1

print("\nReaction extent vector of key reactions:")
for reaction, value in zip(reaction_order[:R_nu], reaction_extent):
    print(f"{reaction}: {value:.3f} mol/s")

print("Delta n_1:")
print(delta_n_1)

print("\nDelta n_2:")
print(delta_n_2)

non_key_components = ["H2", "H2O", "CO"]

print("\nCalculated changes of non-key components:")
for component, value in zip(non_key_components, delta_n_2):
    print(f"{component}: {value:.3f} mol/s")

    


Reaction extent vector of key reactions:
R1: 5.500 mol/s
R2: -2.500 mol/s
R4: 1.000 mol/s
Delta n_1:
[-5.5  1.   1. ]

Delta n_2:
[-11.5   6.5   2.5]

Calculated changes of non-key components:
H2: -11.500 mol/s
H2O: 6.500 mol/s
CO: 2.500 mol/s


The calculated non-key componant changes are:

\begin{equation}
\Delta \dot{\underline{n}}_2 =
\begin{bmatrix}
-11.500 & H_2\\
  6.500 & H_2O\\
  2.500 & CO
\end{bmatrix}
\ \mathrm{mol\,s^{-1}}
\end{equation}

The reaction extent of the key reactions is calculated as:

\begin{equation}
\dot{\underline{\xi}} =
\begin{bmatrix}
 5.500 & R_1\\
-2.500 & R_2\\
 1.000 & R_4
\end{bmatrix}
\ \mathrm{mol\,s^{-1}}
\end{equation}

From this, the total output flow rates are calculated:

\begin{equation}
\dot{\underline{n}}_{out}
=
\dot{\underline{n}}_{in}
+
\Delta \dot{\underline{n}}
\end{equation}

In [15]:
# Combine key and non-key component changes
delta_n = np.concatenate((delta_n_1, delta_n_2))

# Calculate outlet molar flow
# n_out = n_in + Δn
n_out = n_in + delta_n

print("Change of key components Δn_1:")
for component, value in zip(component_order[:R_nu], delta_n_1):
    print(f"{component}: {value:.3f} mol/s")

print("\nCalculated change of non-key components Δn_2:")
for component, value in zip(component_order[R_nu:], delta_n_2):
    print(f"{component}: {value:.3f} mol/s")

print("\nTotal change vector Δn:")
for component, value in zip(component_order, delta_n):
    print(f"{component}: {value:.3f} mol/s")

print("\nOutlet molar flow n_out:")
for component, value in zip(component_order, n_out):
    print(f"{component}: {value:.3f} mol/s")

Change of key components Δn_1:
CO2: -5.500 mol/s
CH3OH: 1.000 mol/s
CH3OCH3: 1.000 mol/s

Calculated change of non-key components Δn_2:
H2: -11.500 mol/s
H2O: 6.500 mol/s
CO: 2.500 mol/s

Total change vector Δn:
CO2: -5.500 mol/s
CH3OH: 1.000 mol/s
CH3OCH3: 1.000 mol/s
H2: -11.500 mol/s
H2O: 6.500 mol/s
CO: 2.500 mol/s

Outlet molar flow n_out:
CO2: 4.500 mol/s
CH3OH: 1.200 mol/s
CH3OCH3: 1.300 mol/s
H2: 18.500 mol/s
H2O: 7.190 mol/s
CO: 4.500 mol/s


The total change in flow of all components is calculated as:

\begin{equation}
\Delta \dot{\underline{n}} =
\begin{bmatrix}
-5.500 & CO_2\\
 1.000 & CH_3OH\\
 1.000 & CH_3OCH_3\\
-11.500 & H_2\\
 6.500 & H_2O\\
 2.500 & CO
\end{bmatrix}
\ \mathrm{mol\,s^{-1}}
\end{equation}

Therefore the output flow of all components is:

\begin{equation}
\dot{\underline{n}}_{out} =
\begin{bmatrix}
 4.500 & CO_2\\
 1.200 & CH_3OH\\
 1.300 & CH_3OCH_3\\
18.500 & H_2\\
 7.190 & H_2O\\
 4.500 & CO
\end{bmatrix}
\ \mathrm{mol\,s^{-1}}
\end{equation}

### 2.5 Plausibility check using atom balance

To verify the integrity of the code and the plausibility of the conducted calculations, an atom balance can be used.
Therefore, atom molar flow vectors $\dot{\underline{b}}$ are created that contain the molar flow rates of each atom species.
 Since chemical reactions do not create or destroy atoms, the total molar flow of each element must be identical at inlet and outlet:

\begin{equation}
\dot{\underline{b}}_{in} =
\begin{bmatrix}
\dot{b}_{C,in}\\
\dot{b}_{H,in}\\
\dot{b}_{O,in}
\end{bmatrix} =
\dot{\underline{b}}_{out} =
\begin{bmatrix}
\dot{b}_{C,out}\\
\dot{b}_{H,out}\\
\dot{b}_{O,out}
\end{bmatrix}
\end{equation}

Also the element-species matrix $\underline{B}$ has to be taken into account. Each of its columns represents a component while each row represents an element of the reaction network. In this case the element-species matrix can be formed as:

\begin{equation}
\underline{B} =
\begin{bmatrix}
      & CO_2 & CH_3OH & CH_3OCH_3 & H_2 & H_2O & CO\\
C     & 1    & 1      & 2         & 0   & 0    & 1\\
H     & 0    & 4      & 6         & 2   & 2    & 0\\
O     & 2    & 1      & 1         & 0   & 1    & 1
\end{bmatrix}
\end{equation}

The inlet and outlet atom flows can be calculated:

\begin{equation}
    \underline{\dot{b}_{in}} = \underline{B \dot{n}_{in}}
\end{equation}

where $\dot{n}_{in}$ is the molar flow vector of all components. Same can be applied to the outlet stream.
To validate the calculation, the following must hold:

\begin{equation}
    \dot{\underline{b}}_{in} = \dot{\underline{b}}_{out} 
\end{equation}


In [17]:
# Element-species matrix B
# rows: C, H, O
# columns: CO2, CH3OH, CH3OCH3, H2, H2O, CO
B = np.array([
    [1, 1, 2, 0, 0, 1],  # C atoms
    [0, 4, 6, 2, 2, 0],  # H atoms
    [2, 1, 1, 0, 1, 1]   # O atoms
], dtype=float)

elements = ["C", "H", "O"]

# Calculate element flows
b_in = B @ n_in # calculation of element flow at inlet using matrix multiplication of B and n_in (@ is the matrix multiplication operator in numpy)
b_out = B @ n_out
b_difference = b_out - b_in

print("\nAtom balance check:")
for element, inlet, outlet, diff in zip(elements, b_in, b_out, b_difference):
    print(
        f"{element}: "
        f"in : {inlet:.6f} mol atoms/s, "
        f"out : {outlet:.6f} mol atoms/s, "
        f"difference : {diff:.2e}"
    )

if np.allclose(b_in, b_out):
    print("\nAtom balance is fulfilled.")
else:
    print("\nAtom balance is NOT fulfilled.")


Atom balance check:
C: in : 12.800000 mol atoms/s, out : 12.800000 mol atoms/s, difference : 0.00e+00
H: in : 63.980000 mol atoms/s, out : 63.980000 mol atoms/s, difference : -7.11e-15
O: in : 23.190000 mol atoms/s, out : 23.190000 mol atoms/s, difference : -3.55e-15

Atom balance is fulfilled.


The atom balance is fullfilled in this case. That means that every atom that makes up the inlet flow is also present in the outlet flow. Therefore the calculation of the conversion is valid.

Optionally, the overall mass balance can be used to verify the calculations. In this case, the molar mass $M_i$ of every component can be used to calculate inlet and outlet mass flowrates $\underline{\dot{m}}_{in}$ and $\underline{\dot{m}}_{out}$.

The overall inlet and outlet mass flows can be calculated:
\begin{equation}
    \dot{m}_{in}
    =
    \sum_{i=1}^{N}
    \dot{n}_{in,i} \, M_i
\end{equation}

\begin{equation}
    \dot{m}_{out}
    =
    \sum_{i=1}^{N}
    \dot{n}_{out,i} \, M_i
\end{equation}

Comparing the mass flowrates, the difference should be (numerically close to) zero.

In [19]:
# Optional mass balance check

# Molar masses in g/mol
# order: CO2, CH3OH, CH3OCH3, H2, H2O, CO
M = np.array([
    44.01,   # CO2
    32.04,   # CH3OH
    46.07,   # CH3OCH3
    2.016,   # H2
    18.015,  # H2O
    28.01    # CO
])

m_in = np.sum(n_in * M)
m_out = np.sum(n_out * M)
m_difference = m_out - m_in

print("\nMass balance check:")
print(f"m_in  = {m_in:.6f} g/s")
print(f"m_out = {m_out:.6f} g/s")
print(f"difference = {m_difference:.2e} g/s")



Mass balance check:
m_in  = 589.259350 g/s
m_out = 589.252850 g/s
difference = -6.50e-03 g/s


Therefore the mass balance should be fulfilled by a margin of error that is not significant for the given inlet and outlet streams.

# Sources

[1] Güttel, R., & Turek, T. (2021). *Chemische Reaktionstechnik*. Springer Spektrum. https://doi.org/10.1007/978-3-662-63150-8